In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS github_lakehouse.audit;

CREATE TABLE IF NOT EXISTS github_lakehouse.audit.silver_pipeline_runs (
    -- Run identity and scope
    audit_id                       STRING,   -- unique per attempt
    run_id                         STRING,   -- Job run id, or manual-<uuid>
    pipeline_name                  STRING,   -- e.g. silver_push_events
    event_type                     STRING,   -- e.g. PushEvent
    processing_scope               STRING,   -- SINGLE_DATE | ALL_DATES
    source_date                    DATE,     -- NULL when ALL_DATES
    run_started_at                 TIMESTAMP,
    run_finished_at                TIMESTAMP,
    status                         STRING,   -- SUCCEEDED | NO_INPUT | FAILED
    error_message                  STRING,

    -- Input classification (one pass over the classified input)
    bronze_input_rows              BIGINT,   -- rows of this event type in scope
    hard_null_quarantine_rows      BIGINT,
    validation_quarantine_rows     BIGINT,
    validated_rows                 BIGINT,
    duplicate_rows_removed         BIGINT,   -- validated - silver candidates

    -- Silver write
    silver_candidate_rows          BIGINT,   -- distinct event_id after dedup
    silver_rows_inserted           BIGINT,
    silver_rows_already_present    BIGINT,
    silver_table_version           BIGINT,

    -- Quarantine write
    quarantine_candidate_rows      BIGINT,   -- distinct quarantine_key
    quarantine_duplicates_removed  BIGINT,
    quarantine_rows_inserted       BIGINT,
    quarantine_table_version       BIGINT,

    -- Event-specific quality counts (denominator: validated_rows)
    quality_metrics                MAP<STRING, BIGINT>,
    reconciliation_passed          BOOLEAN
)
USING DELTA;